# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Blocked by material data availability: the labeled four-class wheat corpus required by the author's Random Forest notebook is not publicly available. The only public data release (Kaggle engrirf/wheat-plants-disease-dataset-for-kp-pakistan v1, 64,936,832 B, verified in Colab) contains 25 unlabeled field photographs with empty VOC XML annotations and a 2-class labelmap; the author's notebook loads a private Drive DATA.zip; Zenodo archives code only; no PMC/OpenAlex copy of the paper exists and PeerJ serves a Cloudflare challenge. A partial executed demonstration (data acquisition, real-image preview, author preprocessing on the 25-image sample) plus a designed RuntimeError training guard was delivered as an UNVERIFIED/INCOMPLETE reference under deliverables/. No publication: the accuracy claims (paper 89%, author run 87.68%) cannot be faithfully re-measured.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Plants disease detection and identification system using AI and computer vision techniques

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-ed7b7c55865a51b68f3cec66b4685c0e)


# Random Forest wheat disease classification — reproduction of the released author notebook

**Paper:** Irfan Ahmed et al., *"Plants disease detection and identification system using AI and computer vision techniques"*, **PeerJ Computer Science 12:e3509** (2026), DOI [10.7717/peerj-cs.3509](https://doi.org/10.7717/peerj-cs.3509) (CC-BY 4.0).

**Author code:** [`engriffi1/wheat-plant-dataset`](https://github.com/engriffi1/wheat-plant-dataset) @ commit `de3e93ff62baadf6daadc3a211f961cb597e2392` (Apache-2.0), file `Random Forest Classification.ipynb`.

## ⚠️ Status: PARTIALLY EXECUTED — training BLOCKED by missing labeled data

| Pipeline stage | Status | Evidence |
| --- | --- | --- |
| Author's Random Forest training/evaluation code | ✅ available (pinned commit) | `Random Forest Classification.ipynb` |
| The labeled four-class wheat corpus (`DATA/train`, `DATA/valid`: *healthy, leaf rust, barley yellow dwarf, powdery mildew*) used by that notebook | ❌ **not publicly available** | the notebook loads a private Drive `DATA.zip` |
| Public Kaggle release (`engrirf/wheat-plants-disease-dataset-for-kp-pakistan`, v1) | ⚠️ **25 unlabeled field photos + empty VOC XML annotations + a 2-class labelmap** | verified in Colab (see §3) |
| Trained weights | ❌ not released | no checkpoint in repo/Zenodo |

This notebook **executes** everything the public materials support — acquiring the paper-linked Kaggle sample, previewing the actual field photographs, and running the author's exact preprocessing — and then **stops with an explicit `RuntimeError`** before training, because retraining and the paper's 89% accuracy (Table 5; the author's own run: 87.68%) cannot be faithfully reproduced without the labeled corpus. The complete training/evaluation cells are included and will run only if the labeled data is supplied.

**日本語:** 本ノートブックは、論文（PeerJ Computer Science 12:e3509）の著者公開 Random Forest パイプラインの再現を試みます。公開されているのは 25 枚のラベルなしサンプル画像（アノテーション空欄）のみで、4 クラスの学習用コーパスは非公開のため、前処理までを実行し、学習の直前で明示的なエラーで停止します（誤解を招く結果を出さないための設計です）。

## 1. Runtime selection

**Select `Runtime → Change runtime type → CPU` (the standard free CPU runtime is sufficient).**
The executed stages (dataset download, image preview, preprocessing) and, if labeled data were supplied, the scikit-learn Random Forest training do not use a GPU. The paper trained its deep models on a Colab Tesla K80, but the Random Forest path does not need one.

**日本語:** 「ランタイム → ランタイムのタイプを変更 → CPU」を選択してください。GPU は不要です。

### 1.1 Environment check

Prints the Python, NumPy, scikit-learn and OpenCV versions actually used by this run, so the executed environment is recorded in the output.

**日本語:** 最初に実際に使用する Python・NumPy・scikit-learn・OpenCV のバージョンを確認します。

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0], "|", platform.machine())

import numpy as np
import sklearn, cv2, matplotlib
print("numpy", np.__version__)
print("scikit-learn", sklearn.__version__)
print("opencv", cv2.__version__)
print("matplotlib", matplotlib.__version__)

### 1.2 Install `kagglehub`

The public dataset is fetched with the official `kagglehub` client (no Kaggle account or API key is needed for this public, non-consent-gated dataset). Colab already ships `kagglehub`; this cell only ensures it is present and prints the version.

**日本語:** 公開データセットの取得には公式の `kagglehub` を使います。アカウント登録は不要です。

In [ ]:
%pip install -q kagglehub

import kagglehub
print("kagglehub", kagglehub.__version__)

## 2. Data acquisition and provenance

Download **version 1** (the only release, "Initial release", 2024-01-02; 64,936,832 bytes per the Kaggle metadata API) of `engrirf/wheat-plants-disease-dataset-for-kp-pakistan` onto the Colab VM. The dataset owner is Irfan Ahmed — the paper's supervisor and corresponding author named in the repository README. Kaggle lists the license field as "Apache 2.0"; the author's README states the data is for **educational use, commercial use prohibited without permission from Dr. Irfan Ahmed**. All dataset bytes stay inside this Colab VM.

**日本語:** Kaggle の公開データセット（バージョン 1、約65MB）を Colab VM 内にダウンロードします。所有者は論文著者（指導教員）の Irfan Ahmed 氏です。教育目的の利用が前提とされています。

In [ ]:
import kagglehub

# Pinned handle with explicit version 1 (the only published version of this dataset)
DATA_ROOT = kagglehub.dataset_download("engrirf/wheat-plants-disease-dataset-for-kp-pakistan/versions/1")
print("Dataset downloaded to:", DATA_ROOT)

## 3. What the public release actually contains

List the extracted tree with file counts and sizes. The release is a Pascal-VOC-style annotation project: `JPEGImages/` (25 photographs), `Annotations/` (25 XML files), `ImageSets/Main/default.txt` (the 25 filenames), and `labelmap.txt`. There are **no class folders and no train/valid split**, i.e. it is not the corpus the author's Random Forest notebook loads.

**日本語:** 展開されたファイル構成を一覧します。Pascal VOC 形式のアノテーションプロジェクトであり、クラス別フォルダや train/valid 分割は含まれていません。

In [ ]:
from pathlib import Path

root = Path(DATA_ROOT)
jpgs = sorted((root / "JPEGImages").glob("*.jpg"))
xmls = sorted((root / "Annotations").glob("*.xml"))
print(f"JPEGImages: {len(jpgs)} files | Annotations: {len(xmls)} files")
total = sum(p.stat().st_size for p in root.rglob("*") if p.is_file())
print(f"total extracted bytes: {total:,}")
for p in sorted(root.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(root)} ({p.stat().st_size:,} B)")
    else:
        print(f"  {p.relative_to(root)}/")

### 3.1 Input preview — the paper's actual field photographs

Display the first eight of the 25 photographs from `JPEGImages/`. These are real crop images from the paper-linked collection (filenames `IMG_20220115_*` match the paper's field campaign period). **No class labels exist for these images in the public release**, so they are captioned by filename only — inventing labels would be fabrication.

**日本語:** 公開されている 25 枚のうち最初の 8 枚を表示します。公開データにはクラスラベルが存在しないため、ファイル名のみをキャプションにします（ラベルの推測は行いません）。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for ax, p in zip(axes.ravel(), jpgs[:8]):
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # matplotlib expects RGB
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
plt.suptitle("Kaggle wheat release v1 — 25 unlabeled field photographs (8 of 25 shown)", fontsize=12)
plt.tight_layout()
plt.savefig("wheat_sample_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"shown 8 of {len(jpgs)} photographs; all {len(jpgs)} are {cv2.imread(str(jpgs[0])).shape[1]}x{cv2.imread(str(jpgs[0])).shape[0]} px (portrait/landscape varies)")

### 3.2 Annotations and labelmap — why training is blocked

Read `labelmap.txt` and `ImageSets/Main/default.txt`, and count `<object>` elements across all XML annotations. Expected result from the verification run: the labelmap declares only `background`, `healthy`, `powdery mildew`; the split file is just the 25 filenames; and **every XML contains zero annotated objects**. The public release therefore provides no per-image class labels and no bounding boxes, and covers only 25 images — versus the 1,490 wheat photographs (Table 2) and four classes (*healthy, leaf rust, barley yellow dwarf, powdery mildew*) that the author's Random Forest notebook requires.

**日本語:** labelmap と XML アノテーションの中身を確認します。検証の結果、labelmap は `healthy` と `powdery mildew` のみ、XML の object 要素はすべて 0 個、分割ファイルはファイル名一覧のみでした。つまり学習に必要なラベルが公開データには一切ありません。

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

print("--- labelmap.txt ---")
print((root / "labelmap.txt").read_text())
print("--- ImageSets/Main/default.txt ---")
print((root / "ImageSets" / "Main" / "default.txt").read_text())

counter = Counter()
for xml_path in xmls:
    tree = ET.parse(xml_path)
    counter.update(o.findtext("name") for o in tree.findall("object"))
print("object class counts across all 25 XMLs:", dict(counter) or "NONE (no <object> elements)")
print("labelmap classes:", [ln.split(":")[0] for ln in (root / "labelmap.txt").read_text().splitlines() if ln])

### 3.3 Author's preprocessing on the public sample (executed faithfully)

Run the author's exact preprocessing pipeline on all 25 photographs: delete non-JFIF JPEGs (the author's notebook deleted 85/29 in their own run), load with `cv2.IMREAD_COLOR`, resize to **224×224**, apply the author's `cv2.cvtColor(img, cv2.COLOR_RGB2BGR)` (kept verbatim; applied identically to all images it is harmless), scale to [0, 1], cast to float32, and flatten each image to the author's 150,528-dimensional feature vector. This proves the preprocessing stage works end to end; what is missing is **labels**, not the pipeline.

**日本語:** 著者の前処理（JFIF チェック、224×224 リサイズ、[0,1] 正規化、150,528 次元への展開）を 25 枚の公開画像に対してそのまま実行します。パイプライン自体は動作しますが、学習に必要なラベルが無いことがボトルネックです。

In [ ]:
import os

SIZE = 224  # author's setting

# 1) JFIF corruption check, as in the author's notebook
num_skipped = 0
for p in jpgs:
    with open(p, "rb") as fobj:
        if b"JFIF" not in fobj.peek(10):
            num_skipped += 1
            os.remove(p)
jpgs = [p for p in jpgs if p.exists()]
print("Deleted %d corrupted (non-JFIF) images" % num_skipped)

# 2) Load, resize, normalize — author's cells, applied to the sample
images = []
for img_path in jpgs:
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (SIZE, SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)  # kept verbatim from the author's notebook
    images.append(img)
images = (np.array(images) / 255.0).astype(np.float32)

# 3) Flatten to the author's feature vectors
samples, x, y, z = images.shape
features = images.reshape((samples, x * y * z))
print("preprocessed sample features:", features.shape, features.dtype)

## 4. Training guard — why this notebook stops here

The author's Random Forest pipeline needs an image-level label per photograph
(`train_labels` in their notebook comes from the four class folder names of
`DATA/train` and `DATA/valid`). No public release contains those labels: the
Kaggle release has no class folders, its XML annotations are empty, and the
Zenodo archive (DOI 10.5281/zenodo.17848526, v1.0.0) contains only the code
notebooks. Training on unlabeled images — or inventing labels — would produce a
scientifically meaningless or fabricated result, so this cell **raises
`RuntimeError` on purpose**. "Run all" stops here; the cells below document the
faithful training/evaluation code and execute only if a labeled corpus is supplied.

**日本語:** 著者の RF パイプラインには画像ごとのクラスラベルが必要ですが、公開資料のどこにも存在しません（Kaggle はラベルなし 25 枚、Zenodo はコードのみ）。無理に学習すると無意味・捏造した結果になるため、このセルは意図的に `RuntimeError` を発生させて停止します。

In [ ]:
labeled_class_dirs = [
    p for p in root.rglob("*")
    if p.is_dir() and p.name.strip().lower() in
       ("barley yellow dwarf", "healthy", "leaf rust", "powdery mildew")
]
if not labeled_class_dirs:
    raise RuntimeError(
        "BLOCKED: the labeled four-class wheat corpus required by the author's "
        "Random Forest notebook (a DATA.zip-style train/valid split with folders "
        "'barley yellow dwarf', 'healthy', 'leaf rust', 'powdery mildew') is not "
        "publicly available. Verified 2026-10-31: the public Kaggle release v1 "
        "(engrirf/wheat-plants-disease-dataset-for-kp-pakistan) contains only 25 "
        "unlabeled photographs with empty VOC XML annotations and a 2-class labelmap; "
        "the Zenodo record (10.5281/zenodo.17848526) archives only the code notebooks. "
        "Retraining and accuracy evaluation cannot be faithfully executed. Supply the "
        "labeled corpus (e.g. mount the authors' DATA.zip in Colab) and rerun to "
        "execute the cells below."
    )

### 4.1 Train the Random Forest (author's exact hyperparameters)

*Runs only after the guard passes.* `RandomForestClassifier(n_estimators=100, random_state=42)` exactly as in the author's notebook and the paper's Table 1, fitted on flattened 224×224×3 features with integer-encoded labels (the author's alphabetical encoding: *barley yellow dwarf*=0, *healthy*=1, *leaf rust*=2, *powdery mildew*=3).

**日本語:** ガード通過時のみ実行。著者ノートブック・論文 Table 1 と同一のハイパーパラメータで学習します。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing

# labels would come from the class folder of each image (author's construction)
le = preprocessing.LabelEncoder()
le.fit(sorted(["barley yellow dwarf", "healthy", "leaf rust", "powdery mildew"]))
# x_train, y_train, x_test, y_test are defined by the labeled-corpus loading above
RF = RandomForestClassifier(n_estimators=100, random_state=42)
RF.fit(x_train, y_train)
print("Trained RandomForest with", RF.n_estimators, "trees on", x_train.shape[0], "samples")

### 4.2 Predict, accuracy, confusion matrix, per-class report

*Runs only after the guard passes.* Reference values if the labeled corpus is the authors' own split: paper Table 5 **89%**; the author's executed notebook **87.68%** with per-class support 92/44/98/42.

**日本語:** ガード通過時のみ実行。論文の参照値は 89%（Table 5）、著者の実行は 87.68% です。

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

y_pred = RF.predict(x_test)
print("Accuracy = ", accuracy_score(y_test, y_pred) * 100, "%")

cm = confusion_matrix(y_test, y_pred, labels=range(len(le.classes_)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title("Random Forest confusion matrix — wheat valid split")
plt.tight_layout()
plt.savefig("rf_confusion_matrix.png", dpi=120)
plt.show()

print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

## 5. Interpretation and validation record

**What was executed and verified (fresh Colab CPU runtime):** Kaggle v1 download (64,936,832 B), its exact layout (25 JPEGs + 25 empty-annotation XMLs + filename split + 2-class labelmap), preview of the paper-linked field photographs, and the author's preprocessing (JFIF clean → 224×224 → [0,1] → 150,528-dim flattening).

**What is blocked and why:** retraining and evaluating the Random Forest — the labeled four-class corpus is not publicly available through any verified route (private Drive `DATA.zip` in the author's notebook; Kaggle release unlabeled; Zenodo code-only). The notebook therefore stops at the designed guard instead of fabricating results.

**Discrepancy found:** the paper's Data Availability statement points to Kaggle for "the complete dataset", but the public Kaggle release v1 (2024-01-02, predating the paper) holds only a 25-image annotation-in-progress sample. This is a paper-vs-release discrepancy, recorded here rather than silently worked around.

**日本語:** 実行・検証済み: Kaggle v1 のダウンロードとレイアウト確認、実際の圃場写真のプレビュー、著者の前処理。 blocked な理由: 学習用の 4 クラス ラベル付きコーパスが公開されていないため。論文の Data Availability と実際の Kaggle 公開物の間に乖離があることを記録しました。

### References

1. Ahmed, I. et al. *Plants disease detection and identification system using AI and computer vision techniques.* PeerJ Computer Science 12:e3509 (2026). https://doi.org/10.7717/peerj-cs.3509
2. `engriffi1/wheat-plant-dataset` @ `de3e93ff62baadf6daadc3a211f961cb597e2392` (Apache-2.0), `Random Forest Classification.ipynb`. https://github.com/engriffi1/wheat-plant-dataset
3. Kaggle: *Wheat Plants Disease Dataset for KP Pakistan* (v1, 2024-01-02). https://www.kaggle.com/datasets/engrirf/wheat-plants-disease-dataset-for-kp-pakistan
4. Zenodo: *engriffi1/wheat-plant-dataset: Plant Disease Dataset* (v1.0.0). https://doi.org/10.5281/zenodo.17848526

**日本語:** 出典は上記の通りです。ソースコードは pinned commit、データは Kaggle v1、アーカイブは Zenodo v1.0.0 を確認しています。